# Creación de Figuras

Este notebook adapta el código de visualización del paper original para utilizar los resultados obtenidos con los datos de parlamentarios chilenos.

## Dependencias

In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
from scipy.stats import spearmanr

# Configuración de estilo
plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12
os.makedirs('./figures', exist_ok=True)

## Figura 2: Accuracy por Capa y Cabeza de Atención

Esta figura muestra la capacidad de cada cabeza de atención para predecir la ideología de los parlamentarios.

In [ ]:
models = ['meta-llama_Llama-2-7b-chat-hf', 'mistralai_Mistral-7B-Instruct-v0.1', 'lmsys_vicuna-7b-v1.5']

for model_name in models:
    path = f"../results_replication/{model_name}_politician_performance.pkl"
    if not os.path.exists(path):
        print(f"Archivo no encontrado: {path}")
        continue
        
    performance = pickle.load(open(path, 'rb'))
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(performance, cmap='viridis', cbar_kws={'label': 'Spearman Correlation'})
    plt.xlabel('Attention Heads')
    plt.ylabel('Layers')
    plt.title(f'Layer-Head Accuracy - {model_name.replace("_", "/")}')
    plt.tight_layout()
    plt.savefig(f"./figures/{model_name}_heatmap.pdf")
    plt.show()
    
    top_indices = np.dstack(np.unravel_index(np.argsort(performance.ravel()), performance.shape))[0][-20:, :][::-1]
    print(f"Top 20 Heads for {model_name}:")
    print(top_indices)

## Figura 3: Resultados de Intervención

Efecto de la manipulación de las activaciones en la ideología percibida de las respuestas generadas.

In [ ]:
# Nota: Para esta figura se requiere haber corrido la sección de Intervention en workbench.ipynb
for model_name in models:
    results_path = f"../results_replication/{model_name}_intervention_results.pkl"
    if not os.path.exists(results_path):
        print(f"Archivo no encontrado: {results_path}")
        continue
        
    results = pickle.load(open(results_path, 'rb'))
    # results format: [k, alpha, topic, model_gen_str]
    df_res = pd.DataFrame(results, columns=['k', 'alpha', 'topic', 'text'])
    
    # Visualización básica de los resultados capturados
    plt.figure(figsize=(10, 6))
    sns.countplot(data=df_res, x='alpha', hue='k')
    plt.title(f'Distribución de muestras de intervención ({model_name})')
    plt.xlabel('Alpha')
    plt.ylabel('Cantidad de muestras')
    plt.show()